# Edmonton Crime Spatial Analysis Project
**Author:** Darryl Kuromi | **Course:** GIS502 | **Date:** March 31, 2026

![Banner](./images/banner.png)

### Project Overview
This project explores some of the modern geospatial tools used to analyse large datasets and produce performant visualizations for providing further insight. Using the Edmonton Police Service historic crime occurrences dataset (250,000+ records from 2023 to present) alongside spatial and demographic data from the City of Edmonton's Open Data Portal, the primary goals were to 1) examine relative neighborhood safety and 2) visualize crime trends over time. 

### The Technical Challenge
The desired visual analyses (hexbins/time-series animation) could not be efficiently performed on a dataset of this size using traditional desktop or cloud GIS platforms:
* **ArcGIS Pro:** The CPU-based rendering tools, such as *Enable Feature Binning* and *Visualize Space Time Cube*, ran too slowly to be useful for exploratory data analysis. 
* **ArcGIS Online (AGOL):** The 'Styles' tab allowed for simple heatmaps and square aggregation binning but lacked customization. Running server-side 'Analysis' tools on a quarter-million points was prohibitively expensive, requiring 326+ credits for a single run.

### The Solution
To bypass these bottlenecks, the project pivoted to an open-source, code-first methodology. Spatial data engineering, spatial joins, and aggregations were handled using **GeoPandas** within a Jupyter Notebook. The processed data was then handed off to **Kepler.gl**, a WebGL-powered visualization framework, to render and animate the large datasets in 3D.

In [ ]:
from pathlib import Path
import json
import os
import warnings

import folium
from folium.plugins import DualMap
import geopandas as gpd
from keplergl import KeplerGl
import numpy as np
import pandas as pd
import requests
from shapely import wkb, wkt

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path(".")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
HTML_DIR = PROJECT_ROOT / "html"

CRIME_PARQUET = PROCESSED_DIR / "edmonton_crimes_all_years.parquet"
CRIME_CSV = PROCESSED_DIR / "edmonton_crimes_all_years.csv"
BOUNDARIES_RAW_PARQUET = RAW_DIR / "edmonton_boundaries_raw.parquet"
BOUNDARIES_RAW_CSV = RAW_DIR / "edmonton_boundaries_raw.csv"
BOUNDARIES_PARQUET = PROCESSED_DIR / "edmonton_boundaries.parquet"
BOUNDARIES_CSV = PROCESSED_DIR / "edmonton_boundaries.csv"
BOUNDARIES_ENRICHED_PARQUET = PROCESSED_DIR / "edmonton_boundaries_enriched.parquet"
POIS_PARQUET = PROCESSED_DIR / "pois.parquet"
POIS_CSV = PROCESSED_DIR / "pois.csv"
POI_BUFFERS_PARQUET = PROCESSED_DIR / "poi_buffers.parquet"
POI_BUFFERS_CSV = PROCESSED_DIR / "poi_buffers.csv"
PETTY_THEFT_FREE_PARQUET = PROCESSED_DIR / "edmonton_no_petty_theft.parquet"
PETTY_THEFT_FREE_CSV = PROCESSED_DIR / "edmonton_no_petty_theft.csv"
CHOROPLETH_PARQUET = PROCESSED_DIR / "df_choropleth_final.parquet"
CHOROPLETH_CSV = PROCESSED_DIR / "df_choropleth_final.csv"
KEPLER_CONFIG_PATH = PROJECT_ROOT / "kepler_config.json"

CRS_WEB = "EPSG:4326"
CRS_ALBERTA_3TM = "EPSG:3776"
CRS_WEB_MERCATOR = "EPSG:3857"
CRS_METRIC = "EPSG:32612"

# Binder-friendly defaults: prefer cached parquet assets and skip duplicate exports.
FORCE_REBUILD = False
EXPORT_CSV_ARTIFACTS = False
RENDER_INTERACTIVE_MAPS = "BINDER_SERVICE_HOST" in os.environ
EXPORT_KEPLER_HTML = False
REQUEST_TIMEOUT_SECONDS = 30

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)


def maybe_export_csv(frame, path):
    if not EXPORT_CSV_ARTIFACTS:
        return

    export_frame = pd.DataFrame(frame).copy()
    if "geometry" in export_frame.columns:
        export_frame["geometry"] = export_frame["geometry"].apply(
            lambda value: value.wkt if hasattr(value, "wkt") else value
        )
    export_frame.to_csv(path, index=False)


def load_crime_dataset(columns=None):
    return pd.read_parquet(CRIME_PARQUET, columns=columns)


def coerce_geometry(frame, crs=CRS_WEB):
    if "geometry" not in frame.columns:
        return frame
    if frame.empty:
        return gpd.GeoDataFrame(frame, geometry="geometry", crs=crs)

    sample = next((value for value in frame["geometry"] if value is not None), None)
    if isinstance(sample, str):
        frame = frame.copy()
        frame["geometry"] = frame["geometry"].apply(wkt.loads)
    elif isinstance(sample, bytes):
        frame = frame.copy()
        frame["geometry"] = frame["geometry"].apply(
            lambda value: wkb.loads(value) if value is not None else None
        )

    if not isinstance(frame, gpd.GeoDataFrame):
        frame = gpd.GeoDataFrame(frame, geometry="geometry", crs=crs)
    elif frame.crs is None:
        frame = frame.set_crs(crs)
    return frame


def repair_polygon_geometries(frame, crs=CRS_WEB):
    frame = coerce_geometry(frame, crs=crs).copy()
    if hasattr(frame.geometry, "make_valid"):
        frame["geometry"] = frame.geometry.make_valid()
    else:
        frame["geometry"] = frame.geometry.buffer(0)
    return frame


In [ ]:
# Data Ingestion & Spatial Standardization

files_old = [
    RAW_DIR / "Historic_Occurrences_CSDP_2023.csv",
    RAW_DIR / "Historic_Occurrences_CSDP_2024.csv",
    RAW_DIR / "Historic_Occurrences_CSDP_2025.csv",
]
file_2026 = RAW_DIR / "Historic_Occurrences_CSDP_2026.csv"
raw_columns = ["Occurrence_Category", "Occurrence_Type_Group", "Date Reported", "x", "y"]

if CRIME_PARQUET.exists() and not FORCE_REBUILD:
    print("Loading processed crime dataset from local cache")
    gdf_all_minified = load_crime_dataset()
else:
    print("Processed crime cache missing or rebuild forced; rebuilding from raw CSV files")

    print("Loading and converting 2023-2025 data (EPSG:3776)")
    df_old = pd.concat([pd.read_csv(path, usecols=raw_columns) for path in files_old], ignore_index=True)
    gdf_old = gpd.GeoDataFrame(
        df_old,
        geometry=gpd.points_from_xy(df_old.x, df_old.y),
        crs=CRS_ALBERTA_3TM,
    )

    print("Loading and converting 2026 data (EPSG:3857)")
    df_2026 = pd.read_csv(file_2026, usecols=raw_columns)
    gdf_2026 = gpd.GeoDataFrame(
        df_2026,
        geometry=gpd.points_from_xy(df_2026.x, df_2026.y),
        crs=CRS_WEB_MERCATOR,
    )

    print("Unifying projections to Web WGS84 (EPSG:4326) and combining")
    gdf_all = pd.concat([gdf_old.to_crs(CRS_WEB), gdf_2026.to_crs(CRS_WEB)], ignore_index=True)

    print("Formatting dataset for Kepler.gl")
    gdf_all["Longitude"] = gdf_all.geometry.x
    gdf_all["Latitude"] = gdf_all.geometry.y
    gdf_all["Date Reported"] = pd.to_datetime(gdf_all["Date Reported"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    gdf_all_minified = gdf_all.drop(columns=["geometry", "x", "y"], errors="ignore").copy()

    print("Saving processed master dataset to local cache")
    gdf_all_minified.to_parquet(CRIME_PARQUET, engine="pyarrow", index=False)
    maybe_export_csv(gdf_all_minified, CRIME_CSV)

print(f"Crime dataset ready with {len(gdf_all_minified):,} records and {len(gdf_all_minified.columns)} columns.")


### Data Ingestion & Spatial Standardization

This cell acts as a simple ETL pipeline for the raw crime occurrences. It performs several tasks to prepare the data for web visualization:

* **Handling Two Source Data CRS's:** The raw data changed coordinate reference systems (CRS) between years. The 2023-2025 data was published in Alberta 3TM (`EPSG:3776`), while the 2026 data shifted to Web Mercator (`EPSG:3857`). 
* **Projection Unification:** Both datasets are re-projected to the global GPS standard (`EPSG:4326` - WGS84) so they can be accurately plotted on a web map.
* **Kepler.gl Optimization:** Pure `Latitude` and `Longitude` columns are extracted, and temporal columns are formatted to enable Kepler's time-series animation slider. 
* **Minification & Caching:** Heavy, unnecessary spatial columns are dropped, and the final unified dataset is exported as a lightweight `.parquet` file.

![2023-2025 raw data](./images/image1b.png)
*Figure 1: 2023-2025 raw data in EPSG: 3776 (NAD83 / Alberta 3TM)

![2026 raw data](./images/image1a.png)
*Figure 2: 2026 raw data in EPSG: 3857 (WGS 84 Web Mercator) 

![Unified data](./images/image1d.png)
*Figure 3: 2023-2026 processed data in EPSG: 4326 (WGS 84) 

In [ ]:
# Boundary Data Ingestion & Caching

if BOUNDARIES_PARQUET.exists() and not FORCE_REBUILD:
    print("Loading processed boundary data from local cache")
    gdf_boundaries = gpd.read_parquet(BOUNDARIES_PARQUET)
else:
    if BOUNDARIES_RAW_PARQUET.exists():
        print("Loading raw boundary data from local cache")
        gdf_boundaries_raw = gpd.read_parquet(BOUNDARIES_RAW_PARQUET)
    else:
        print("Fetching raw boundary data from City of Edmonton API")
        geojson_url = "https://data.edmonton.ca/api/geospatial/65fr-66s6?method=export&format=GeoJSON"
        gdf_boundaries_raw = gpd.read_file(geojson_url)

        print("Saving raw boundary data to local cache")
        gdf_boundaries_raw.to_parquet(BOUNDARIES_RAW_PARQUET, engine="pyarrow", index=False)
        maybe_export_csv(gdf_boundaries_raw, BOUNDARIES_RAW_CSV)

    print("Selecting boundary columns needed downstream")
    gdf_boundaries = gdf_boundaries_raw[["neighbourhood_number", "descriptive_name", "geometry"]].copy()

    print("Saving processed boundary data to local cache")
    gdf_boundaries.to_parquet(BOUNDARIES_PARQUET, engine="pyarrow", index=False)
    maybe_export_csv(gdf_boundaries, BOUNDARIES_CSV)

print(f"Boundary dataset ready with {len(gdf_boundaries):,} neighbourhood polygons.")


### Boundary Data Ingestion & Caching

This block handles the retrieval and optimization of the City of Edmonton neighborhood boundary polygons. It used a cache-first pattern to avoid a slow API call if the raw data is present:

* **API Integration:** The script checks for a local copy of the raw data. If it doesn't exist for amy reason (i.e. cloned repo), it fetches the latest GeoJSON boundaries from the City of Edmonton's Open Data API. Datasets are immediately cached to the local disk as `.parquet` files. 
* **Feature Selection:** The raw API data contains many unneeded columns. The script isolates only the three strictly necessary features (`neighbourhood_number`, `descriptive_name`, and `geometry`) to create a lightweight, optimized GeoDataFrame that will perform spatial joins as fast as possible.

A Kelper hexbin plot of the current dataset shows an unexpectedly extreme concentration of crime distribution at several points around the city, in addition to the expected concentration in the downtown core. These likely represent noise that is distorting the true picture.

![Crime Distribution](./images/image2a.png)
*Figure 4: 3D hexbin extrusion of all criminal occurances (2023 to March 2026), s

In [ ]:
# Hotspot Analysis & Ground Truthing (Find the top 8 exact coordinate crime locations)

gdf_all = load_crime_dataset(columns=["Latitude", "Longitude", "Occurrence_Type_Group"])

top_locations = (
    gdf_all.groupby(["Latitude", "Longitude"])
    .size()
    .reset_index(name="Total_Crimes")
    .sort_values(by="Total_Crimes", ascending=False)
    .head(8)
)

print("Crime Hotspot Coordinates (greater than 1000 crimes) with top crime types:")

for rank, (_, row) in enumerate(top_locations.iterrows(), start=1):
    lat = row["Latitude"]
    lon = row["Longitude"]
    total = row["Total_Crimes"]
    spike_data = gdf_all[(gdf_all["Latitude"] == lat) & (gdf_all["Longitude"] == lon)]

    print(f"\n#{rank} | Lat: {lat}, Lon: {lon} | Total Crimes: {total}")
    print(spike_data["Occurrence_Type_Group"].value_counts().head(5).to_string())
    print("-" * 50)


### Hotspot Analysis & Ground Truthing
This cell analyses the micro-level point clusters. It identifies the exact geographic coordinates with the highest absolute concentration (greater than 1000 crimes) of historical incidents:

* **Point-Level Aggregation:** Uses Pandas to group and count incidents by exact Latitude and Longitude, isolating the top 8 busiest coordinate pairs in the entire city.
* **Incident Context:** Iterates through these highly localized spikes to extract and rank the specific *types* of crimes driving the volume at each specific location.
* **Real-World Ground Truthing:** Contextualizes the raw coordinate data with physical landmarks. This manual verification reveals a massive analytical insight: 7 of the 8 highest-crime intersections in Edmonton correspond directly to large retail grocery stores (Real Canadian Superstore), overwhelmingly driven by "Theft Under $5000" (shoplifting).

Rank | Location | Total Crimes |
| :--- | :--- | :--- |
| 1 | Kingsway Real Canadian Superstore | 3,070 |
| 2 | West Edmonton Mall | 2122 |
| 3 | Castledowns Real Canadian Superstore | 2107 |
| 4 | Calgary Trail Real Canadian Superstore | 1848 |
| 5 | Castledowns Real Canadian Superstore (Secondary) | 1513 |
| 6 | 137th Ave Real Canadian Superstore | 1309 |
| 7 | Stony Plain Rd Real Canadian Superstore | 1112 |
| 8 | South Common Real Canadian Superstore | 1026 |


In [ ]:
# Edmonton Superstore Mapping and Buffer Creation

pois = [
    [53.59933409657834, -113.41140526930225, "Real Canadian Superstore"],
    [53.60155846148393, -113.53701529641887, "Real Canadian Superstore"],
    [53.564754391613455, -113.52167267996755, "Real Canadian Superstore"],
    [53.54005171995545, -113.62122967702152, "Real Canadian Superstore"],
    [53.48609373990756, -113.49375711677843, "Real Canadian Superstore"],
    [53.45207032994878, -113.48115709548165, "Real Canadian Superstore"],
    [53.48061778133973, -113.37410282218761, "Real Canadian Superstore"],
    [53.42746836323237, -113.42466595239625, "Real Canadian Superstore"],
    [53.43715350352188, -113.6176286913236, "Real Canadian Superstore"],
]

if POIS_PARQUET.exists() and POI_BUFFERS_PARQUET.exists() and not FORCE_REBUILD:
    print("Loading POIs and buffers from local cache")
    gdf_pois = coerce_geometry(gpd.read_parquet(POIS_PARQUET))
    df_poi_buffers = pd.read_parquet(POI_BUFFERS_PARQUET)
    gdf_poi_buffers = coerce_geometry(df_poi_buffers)
else:
    print("Building POIs and 450m buffers")
    gdf_pois = gpd.GeoDataFrame(
        pois,
        columns=["Latitude", "Longitude", "Location_Type"],
        geometry=gpd.points_from_xy([row[1] for row in pois], [row[0] for row in pois]),
        crs=CRS_WEB,
    )

    gdf_pois.to_parquet(POIS_PARQUET, engine="pyarrow", index=False)
    maybe_export_csv(gdf_pois, POIS_CSV)

    print("Projecting to meters & drawing buffers")
    gdf_pois_meters = gdf_pois.to_crs(CRS_METRIC)
    gdf_poi_buffers = gdf_pois_meters.copy()
    gdf_poi_buffers["geometry"] = gdf_poi_buffers.geometry.buffer(450)
    gdf_poi_buffers = gdf_poi_buffers.to_crs(CRS_WEB)

    df_poi_buffers = pd.DataFrame(gdf_poi_buffers).copy()
    df_poi_buffers["geometry"] = gdf_poi_buffers.geometry.to_wkt()
    df_poi_buffers.to_parquet(POI_BUFFERS_PARQUET, engine="pyarrow", index=False)
    maybe_export_csv(df_poi_buffers, POI_BUFFERS_CSV)

print(f"POI buffers ready for {len(gdf_poi_buffers):,} locations.")


### Edmonton Superstore Mapping and Buffer Creation
Following the ground-truthing discoveries, this cell generates buffers around every Real Canadian Superstore in Edmonton to analyze localized crime density.

* **POI Mapping:** Google Maps was used to determine the exact coordinates of each location.
* **Re-projection for Spatial Math):** To draw an accurate 450-meter buffer (the smallest radius which captures all the nearest intersection points), the points are temporarily projected from the raw GPS coordinates (`EPSG:4326`) into a portable metric CRS for the Edmonton area (`EPSG:32612`, WGS 84 / UTM zone 12N). Once the buffers are drawn, they are projected *back* to global GPS coordinates for web mapping.
* **Web-Safe Serialization (WKT):** Converts the complex 2D polygon objects into Well-Known Text (WKT) strings. This is a critical step that prevents binary encoding crashes when passing spatial data from Python `.parquet` files directly into JavaScript-based tools like Kepler.gl.


In [ ]:
# Analyze Proportion of Shoplifting Crime Within Buffer 

gdf_all = load_crime_dataset(columns=["Occurrence_Type_Group", "Longitude", "Latitude"])
gdf_poi_buffers = repair_polygon_geometries(pd.read_parquet(POI_BUFFERS_PARQUET))
gdf_boundaries = repair_polygon_geometries(gpd.read_parquet(BOUNDARIES_PARQUET))

shoplifting_crimes = gdf_all[gdf_all["Occurrence_Type_Group"] == "Theft Under $5000"].copy()
shoplifting_gdf = gpd.GeoDataFrame(
    shoplifting_crimes,
    geometry=gpd.points_from_xy(shoplifting_crimes["Longitude"], shoplifting_crimes["Latitude"]),
    crs=CRS_WEB,
)

total_shoplifting = len(shoplifting_gdf)
shoplifting_in_buffers = gpd.sjoin(shoplifting_gdf, gdf_poi_buffers, how="inner", predicate="intersects")
buffer_shoplifting_count = len(shoplifting_in_buffers)

buffers_metric = repair_polygon_geometries(gdf_poi_buffers.to_crs(CRS_METRIC), crs=CRS_METRIC)
city_metric = repair_polygon_geometries(gdf_boundaries.to_crs(CRS_METRIC), crs=CRS_METRIC)

buffer_area_sq_km = buffers_metric.union_all().area / 1_000_000
city_area_sq_km = city_metric.geometry.area.sum() / 1_000_000

crime_percentage = (buffer_shoplifting_count / total_shoplifting) * 100
area_percentage = (buffer_area_sq_km / city_area_sq_km) * 100

print("--- SHOPLIFTING POI BUFFER ANALYSIS ---")
print(f"Total Shoplifting Incidents City-Wide: {total_shoplifting:,}")
print(f"Shoplifting Incidents within POI 450m Buffers: {buffer_shoplifting_count:,}")
print(f"Percentage Concentrated near POIs: {crime_percentage:.2f}%\n")

print("--- SPATIAL CONCENTRATION ---")
print(f"Total City Area: {city_area_sq_km:.2f} sq km")
print(f"Total Buffer Area: {buffer_area_sq_km:.2f} sq km")
print(f"Percentage of City Area Occupied by Buffers: {area_percentage:.2f}%")
print(f"\nINSIGHT: {crime_percentage:.1f}% of all shoplifting occurs in less than {area_percentage:.1f}% of the city's landmass.")


### Analyze Proportion of Shoplifting Crime Within Buffer
This cell compares the volume of shoplifting caught inside the 450-meter buffer zones against the total landmass those buffers occupy, proving a massive spatial disparity.

* **Point-in-Polygon Join:** Filters the entire city's crime dataset exclusively for "Theft Under $5000" and uses a spatial join (`sjoin`) to mathematically capture only the dots that fall inside the buffer polygons.
* **Topological Dissolve (`union_all`):** Dissolves the store buffers into one continuous polygon before calculating area, which avoids double-counting the overlapping buffer footprints.
* **Metric Area Calculation:** Projects both the city boundary and the buffers into a portable metric CRS for Edmonton (`EPSG:32612`) so the area calculation stays Binder-compatible.
* **The Proportion Calculation:** Calculates the proportion of the crime density versus the geographic footprint.

> **Conclusion:** 23.9% of all shoplifting in Edmonton occurs in less than 0.7% of the city's total area.


In [ ]:
# New Dataset: Excluding Theft Under $5000

gdf_all = load_crime_dataset()
original_count = len(gdf_all)
crimes_to_ignore = ["Theft Under $5000"]

print("Filtering out petty theft")
df_no_petty_theft = gdf_all[~gdf_all["Occurrence_Type_Group"].isin(crimes_to_ignore)].copy()
new_count = len(df_no_petty_theft)

print("-" * 30)
print(f"Original total crimes: {original_count}")
print(f"Crimes after dropping shoplifting: {new_count}")
print(f"Total rows removed: {original_count - new_count}")
print("-" * 30)

print("Saving the neighborhood-focused dataset")
df_no_petty_theft.to_parquet(PETTY_THEFT_FREE_PARQUET, engine="pyarrow", index=False)
maybe_export_csv(df_no_petty_theft, PETTY_THEFT_FREE_CSV)


### New Dataset: Excluding Theft Under $5000

Following the discovery that massive spikes in retail shoplifting ("Theft Under $5000") were heavily skewing the city-wide density maps, this cell recalibrates the dataset. 

* **Petty Crime Removal:** Strategically filters out petty retail theft to prevent commercial grocery stores from masking the true distribution of violent or property crimes in residential neighborhoods.
* **Exclusion Logic:** Utilizes Pandas boolean indexing (`~isin()`) to rapidly drop the targeted crime categories across the entire quarter-million-row dataset.
* **Dataset Branching:** Instead of overwriting the master dataset, this step exports a dedicated, secondary dataset (`edmonton_no_petty_theft.parquet`). This allows the final dashboard to independently analyze both "Total Crime" and "Filtred Crime" without the two metrics polluting each other.

The same Kelper hexbin plot using the filtered dataset shows a more uniform and predictable distribution of crime, with a single peak near West Edmonton Mall and the expected concentration downtown. However, this drops nearly a third of all data points and in a very arbitrary and simplistic manner. A better method is needed. 

![Petty Theft Free Crime Distribution](./images/image2b.png)
*Figure 5: 3D extrusion of criminal occurances excluding theft under $5000, removing the spikes associated with the RCSS locations but leaving the spike  around the city in addition to the expected concentration in the downtown core.

In [ ]:
# Neighbourhood Data Enrichment: Population & Land Area

if BOUNDARIES_ENRICHED_PARQUET.exists() and not FORCE_REBUILD:
    print("Enriched neighbourhood boundary data found - loading directly from local cache")
    gdf_boundaries_enriched = gpd.read_parquet(BOUNDARIES_ENRICHED_PARQUET)
else:
    print("Building enriched neighbourhood boundaries")
    gdf_boundaries = coerce_geometry(gpd.read_parquet(BOUNDARIES_PARQUET))

    print("Calculating neighborhood areas (Sq Km)")
    gdf_boundaries = gdf_boundaries.to_crs(CRS_METRIC)
    gdf_boundaries["area_sq_km"] = gdf_boundaries.geometry.area / 1_000_000
    gdf_boundaries = gdf_boundaries.to_crs(CRS_WEB)

    print("Fetching 2021 Census population data from City of Edmonton API")
    pop_api_url = "https://data.edmonton.ca/resource/eg3i-f4bj.json"
    response = requests.get(pop_api_url, params={"$limit": 1000}, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    df_pop = pd.DataFrame(response.json())

    df_pop["neighbourhood"] = df_pop["neighbourhood"].replace({"Oliver": "Wîhkwêntôwin"})
    df_pop["neighbourhood_number"] = df_pop["neighbourhood_number"].replace({"1150": "1151"}).astype(str)
    df_pop["total_population"] = pd.to_numeric(df_pop["total_population"], errors="coerce").fillna(0)
    df_pop = df_pop[["neighbourhood_number", "total_population"]]

    print("Joining population to geometries")
    gdf_boundaries["neighbourhood_number"] = gdf_boundaries["neighbourhood_number"].astype(str)
    gdf_boundaries_enriched = gdf_boundaries.merge(df_pop, on="neighbourhood_number", how="left")
    gdf_boundaries_enriched["total_population"] = gdf_boundaries_enriched["total_population"].fillna(0)

    print("Saving fully enriched boundaries to local cache")
    gdf_boundaries_enriched.to_parquet(BOUNDARIES_ENRICHED_PARQUET, engine="pyarrow", index=False)

    print(f"Success! Processed {len(gdf_boundaries_enriched)} neighborhoods with population and area.")

print("\nEnriched Boundary Head:")
print(gdf_boundaries_enriched[["descriptive_name", "total_population", "area_sq_km"]].head())


### Neighbourhood Data Enrichment: Population & Land Area
This cell upgrades the base neighborhood boundaries with demographic and geographic data. This will allow the final deliverable to go from mapping *total* crime volume to calculating *crime density* (crimes per square kilometer) or *per-capita rates*.

* **Spatial Area Calculation:** Temporarily projects the polygons into a portable metric CRS for Edmonton (`EPSG:32612`) to calculate area in square kilometers before returning them to web-safe GPS coordinates.
* **API Integration:** Fetches the most recent 2021 Census population data directly from the City of Edmonton's JSON API.
* **Real-World Data Wrangling:** Handles historical name changes. Because the census data is from 2021, it uses the outdated name for the "Oliver" neighborhood. The script programmatically intercepts and updates the name to "Wîhkwêntôwin" and more importantly patches the neighborhood ID (which was incremented by one) so the spatial join doesn't drop the data.
* **Data Merging & Caching:** Merges the cleaned population metrics into the spatial dataset, handles missing values, and caches the final "Enriched" master file locally to optimize future runs.


In [ ]:
# The Crime Severity Index

eps_severity_weights = {
    # TIER 1: Extreme Violence & Loss of Life (Max Weight)
    'Homicide': 3000.0,
    
    # TIER 2: Severe Violence, Weapons, & High Public Danger
    'Robbery Personal': 300.0,
    'Robbery Commercial': 300.0,
    'Assault': 300.0,
    'Weapons Complaint Firearm': 300.0,
    'Fire Arson': 300.0,
    'Bomb Threat': 300.0,
    
    # TIER 3: Severe Property/Psychological Harm & Extreme Recklessness
    'Break and Enter Residential': 100.0, # Weighted higher than commercial due to psychological trauma
    'Impaired Driving': 100.0,
    'Criminal Flight Event': 80.0,        # Police chases are highly dangerous to the public
    'Break and Enter Commercial': 75.0,
    
    # TIER 4: Serious Property Loss, Drugs, & General Weapons
    'Weapons Complaint': 50.0,            # Non-firearm (knives, bear spray)
    'Theft of Motor Vehicle': 50.0,
    'Drugs': 50.0,                        # Trafficking/Possession
    'Fraud - Financial': 40.0,
    'Fraud Personal': 40.0,
    'Fraud General': 40.0,
    'Internet Fraud': 40.0,
    'Technology/Internet Crime': 40.0,
    
    # TIER 5: Moderate Property Crimes & Public Indecency
    'Theft Over $5000': 30.0,
    'Possession Stolen Property': 30.0,
    'Counterfeit Money': 20.0,
    'Dangerous Condition': 20.0,
    'Indecent Act': 20.0,
    
    # TIER 6: Petty Theft, Minor Damage, & Public Mischief
    'Theft Under $5000': 10.0,
    'Property Damage': 10.0,
    'Mischief - Property': 10.0,
    'Public Mischief': 10.0,              # Usually false police reports/wasting time
    
    # TIER 7: Nuisance, Social Disorder, & Suspicious Activity
    'Graffiti': 5.0,
    'Disturbance': 3.0,
    'Trouble with Person': 3.0,
    'Suspicious Person': 3.0,
    'Suspicious Vehicle': 3.0,
    'Trespassing': 3.0,
    'Dispute': 3.0,
    'Intoxicated Person': 2.0,
    'Liquor Act': 2.0,
    
    # TIER 8: Administrative, By-Law, & Non-Criminal Occurrences
    'Labour Dispute': 1.0,                # Picket lines are legal unless they escalate
    'Public Health Act': 1.0,
    'Recovered Motor Vehicle': 1.0,       # The theft was the crime; the recovery is just paperwork
    'Abandoned Vehicle': 1.0,
    'Workplace Accident': 1.0             # Investigated by police, but usually an OHS matter
}


### The Crime Severity Index
While filtering out petty theft (in the previous step) give a more accurate picture (real crime clusters Downtown, Whyte Avenue, 118th Ave, Millwoods, etc.), it introduced analytical bias by simply discarding valid data. However, the alternative—keeping it in and treating a stolen candy bar equally to an armed robbery was flawed. A criminologically sound methodology was needed to measure actual societal harm, rather than raw incident volume.

The standard for this is the **Cambridge Crime Harm Index (CHI)**, but no such weighting matrix existed for the Edmonton Police Service's specific terminology. To solve this, AI was utilized to generate a baseline weighting scheme, mapping the unique EPS crime categories to the CHI principles. That baseline was then manually adjusted to up-weight profound physical and psychological trauma (e.g., Homicides adjusted from 1,000 to 3,000 and violent/weapons incidents adjusted from 150-200 to 300).

This methodology prevents high-volume nuisance areas from appearing more dangerous on the map than a neighborhood suffering from a low volume of severe, violent crime.

* **Harm-Based Metric Creation:** Instead of counting `1 crime = 1 point`, this dictionary assigns exponentially scaling numerical values (from 1 to 3,000) to each incident based on the actual danger posed to the public. 
* **AI-Assisted Criminology:** Leveraging an LLM to adapt the globally recognized Cambridge CHI to a unique municipal data schema, creating an unbiased, 8-tier severity scale.
* **Preparation for Spatial Aggregation:** By mapping this dictionary onto the main dataset, the final delliverable can calculate a normalized neighborhood risk score, allowing for a more accurate comparison of relative safety across the city without having to delete any underlying data.

In [ ]:
# Calculating True Neighborhood Risk

if CHOROPLETH_PARQUET.exists() and not FORCE_REBUILD:
    print("Loading final neighbourhood risk layer from local cache")
    df_choropleth = pd.read_parquet(CHOROPLETH_PARQUET)
    gdf_choropleth = coerce_geometry(df_choropleth)
else:
    gdf_all = load_crime_dataset(columns=["Occurrence_Type_Group", "Longitude", "Latitude"])
    gdf_all["Harm_Weight"] = gdf_all["Occurrence_Type_Group"].map(eps_severity_weights).fillna(1.0).astype(float)

    gdf_boundaries = repair_polygon_geometries(gpd.read_parquet(BOUNDARIES_ENRICHED_PARQUET))

    print("Preparing the Crime Points...")
    gdf_crimes = gpd.GeoDataFrame(
        gdf_all,
        geometry=gpd.points_from_xy(gdf_all.Longitude, gdf_all.Latitude),
        crs=CRS_WEB,
    )

    print("Executing the Spatial Join (Point-in-Polygon)")
    crimes_in_neighborhoods = gpd.sjoin(gdf_crimes, gdf_boundaries, how="inner", predicate="within")

    print("Calculating Crime Statistics per Neighborhood")
    crime_stats = crimes_in_neighborhoods.groupby("descriptive_name").agg(
        Total_Incidents=("Harm_Weight", "size"),
        Total_Harm_Score=("Harm_Weight", "sum"),
    ).reset_index()

    print("Merging Stats back to Boundaries...")
    gdf_choropleth = gdf_boundaries.merge(crime_stats, on="descriptive_name", how="left")
    gdf_choropleth[["Total_Incidents", "Total_Harm_Score"]] = gdf_choropleth[["Total_Incidents", "Total_Harm_Score"]].fillna(0)

    population = gdf_choropleth["total_population"].replace(0, np.nan)
    gdf_choropleth["Incidents_Per_Capita"] = gdf_choropleth["Total_Incidents"] / population
    gdf_choropleth["Harm_Per_Capita"] = gdf_choropleth["Total_Harm_Score"] / population
    gdf_choropleth[["Incidents_Per_Capita", "Harm_Per_Capita"]] = gdf_choropleth[["Incidents_Per_Capita", "Harm_Per_Capita"]].replace([np.inf, -np.inf], 0)

    print("Formatting for Kepler.gl")
    df_choropleth = pd.DataFrame(gdf_choropleth).copy()
    df_choropleth["geometry"] = gdf_choropleth.geometry.to_wkt()
    df_choropleth.to_parquet(CHOROPLETH_PARQUET, engine="pyarrow", index=False)
    maybe_export_csv(df_choropleth, CHOROPLETH_CSV)

print("\nSpatial Join Complete. Top Neighborhoods by Total Crime:")
print("\nSorted by Total_Incidents:")
print(gdf_choropleth[["descriptive_name", "Total_Incidents", "Incidents_Per_Capita"]].sort_values(by="Total_Incidents", ascending=False).head())
print("\nSorted by Incidents_Per_Capita:")
print(gdf_choropleth[["descriptive_name", "Total_Incidents", "Incidents_Per_Capita"]].sort_values(by="Incidents_Per_Capita", ascending=False).head())

print("\nTop Neighborhoods by Total Harm Score:")
print("\nSorted by Total_Harm_Score:")
print(gdf_choropleth[["descriptive_name", "Total_Harm_Score", "Harm_Per_Capita"]].sort_values(by="Total_Harm_Score", ascending=False).head())
print("\nSorted by Harm_Per_Capita:")
print(gdf_choropleth[["descriptive_name", "Total_Harm_Score", "Harm_Per_Capita"]].sort_values(by="Harm_Per_Capita", ascending=False).head())

gdf_quick_view = gdf_choropleth.copy()


### Calculating True Neighborhood Risk
This cell takes the disparate datasets—the weighted crime points and the enriched neighborhood polygons and fuses them to calculate actionable, normalized safety metrics.

* **Spatial Rebuilding:** Before joining, the script employs robust checks to ensure all geometries are valid. It dynamically decodes raw binary geometries (WKB) and text geometries (WKT) back into shapely objects, preventing pipeline crashes.
* **Point-in-Polygon Join (`sjoin`):** Executes a massive spatial join, mathematically evaluating hundreds of thousands of individual crime coordinates to determine exactly which neighborhood polygon they occurred within.
* **Two Metric Aggregation:** Groups the newly mapped data to calculate two distinct metrics per neighborhood: 
    1. **Volume:** `Total_Incidents` (Raw count of events).
    2. **Severity:** `Total_Harm_Score` (The sum of the engineered Cambridge CHI weights).
* **Per-Capita Normalization:** Raw counts are heavily skewed by population size. To reveal true comparative risk, the script divides both the incident counts and the harm scores by the most recent (2021) Census population data, yielding *per-capita* rates.
* **Web-Safe Serialization:** Finally, the complex polygons are downgraded back into Well-Known Text (WKT) strings. This guarantees the `.parquet` file can be read natively by Kepler.gl's JavaScript engine without throwing JSON encoding errors.

> By calculating **Per Capita Harm**, the project can now produce useful intelligence for one of the main project goals: *to examine relative neighborhood safety*. 

In [ ]:
# Visualizing the Impact: Synchronized Dual Choropleth

if "gdf_choropleth" not in globals():
    gdf_choropleth = coerce_geometry(pd.read_parquet(CHOROPLETH_PARQUET))

gdf_choropleth = gdf_choropleth.copy()
gdf_choropleth["Harm_Per_Capita"] = gdf_choropleth["Harm_Per_Capita"].round(2)
gdf_choropleth.loc[gdf_choropleth["total_population"] == 0, ["Total_Incidents"]] = np.nan

if RENDER_INTERACTIVE_MAPS:
    dual_map = DualMap(location=[53.5461, -113.4938], zoom_start=11, tiles="CartoDB positron")

    gdf_choropleth.explore(
        column="Total_Incidents",
        cmap="YlOrRd",
        tooltip=["descriptive_name", "Total_Incidents"],
        m=dual_map.m1,
        legend_kwds={"caption": "Total Crime Volume"},
        name="Crime Volume",
        scheme="NaturalBreaks",
        k=12,
        missing_kwds={"color": "#e0e0e0", "label": "Non-Residential / No Data"},
    )

    gdf_choropleth.explore(
        column="Harm_Per_Capita",
        cmap="YlOrRd",
        tooltip=["descriptive_name", "Harm_Per_Capita"],
        m=dual_map.m2,
        legend_kwds={"caption": "Harm Score (Per Capita)"},
        name="Harm Index",
        scheme="NaturalBreaks",
        k=12,
        missing_kwds={"color": "#e0e0e0", "label": "Non-Residential (No Pop.)"},
    )

    dual_map
else:
    print("Interactive Folium rendering is off. Run this on Binder or set RENDER_INTERACTIVE_MAPS = True to display the synchronized map.")


### Visualizing the Impact: Synchronized Dual Choropleth
This cell quickly and simply visualizes the current state of the data and feature engineering pipeline. It uses Folium to generate an interactive, synchronized Dualmap directly within the notebook, showing where the raw crime volume metric is misleading.

* **Data Masking:** Before mapping, the script actively masks out non-residential areas (industrial parks or commercial sectors where population equals zero) in the 'Total Incidents' map to match the 'Per Capita Harm' map, where a score can't be generated (by setting their values to `NaN`). These are then represented in a neutral grey, preventing them from distorting the per-capita color scale.
* **Synchronized A/B Comparison:** Uses `DualMap` to lock controls of two maps together, allowing for a visual comparison. The Left Map (`Total_Incidents`) displays the raw volume of crime. This map essentially highlights "where the police spend the most time" (skewed by high-density retail corridors and downtown nightlife). The Right Map (`Harm_Per_Capita`) displays the severity index normalized by population. This map reveals a more accurate underlying risk to residents, highlighting completely different neighborhoods that suffer from lower-volume but more violent or traumatic incidents.

> This interactive widget allows users to distinguish between high-nuisance crime areas, and high-severity crime areas. It shows the McCauley neighbourhood to have the highest density of serious crime far surpassing the Downtown neighbourhood. Rural North East Horse Hill is a close second, previously appearing to be among the safest districts in Edmonton!

![Kepler Choropleth](./images/image3a.png)
*Figure 6: Kepler 2D and 3D Choropleth (Harm Per Capita)

In [ ]:
# The Deliverable: A Performant Interactive Webmap

print("Loading Binder-friendly cached datasets")
gdf_all = load_crime_dataset(columns=["Latitude", "Longitude", "Date Reported", "Occurrence_Type_Group"])
gdf_pois = coerce_geometry(gpd.read_parquet(POIS_PARQUET))
gdf_boundaries = coerce_geometry(gpd.read_parquet(BOUNDARIES_PARQUET))
df_poi_buffers = pd.read_parquet(POI_BUFFERS_PARQUET)
df_choropleth = pd.read_parquet(CHOROPLETH_PARQUET).fillna(0)
df_choropleth.replace([np.inf, -np.inf], 0, inplace=True)

with open(KEPLER_CONFIG_PATH, "r", encoding="utf-8") as f:
    edmonton_config = json.load(f)

print(f"Crime points: {len(gdf_all):,} | POIs: {len(gdf_pois):,} | Neighborhoods: {len(gdf_boundaries):,}")

if RENDER_INTERACTIVE_MAPS:
    print("Building the Kepler map")
    map_1 = KeplerGl(
        height=800,
        data={
            "zh06rv": gdf_all,
            "7r0ykh": gdf_pois,
            "-4bglt3": gdf_boundaries,
            "poi_buffers": df_poi_buffers,
            "df_choropleth": df_choropleth,
        },
        config=edmonton_config,
        show_docs=False,
    )

    if EXPORT_KEPLER_HTML:
        map_1.save_to_html(file_name=str(HTML_DIR / "edmonton_crime.html"))

    map_1
else:
    print("Interactive Kepler rendering is off. Run this on Binder or set RENDER_INTERACTIVE_MAPS = True to build the dashboard widget.")


![Kepler Choropleth](./images/image3.png)
*Figure 7: Neighbourhood Harm Per Capita Choropleth using Kepler

### The Deliverable: A Performant Interactive Webmap
This final cell initializes the project's primary deliverable: a high-fidelity, WebGL-powered 3D dashboard using **Kepler.gl**. It represents the shift from static analysis to a dynamic, multi-layered exploratory tool capable of rendering the entire 250,000+ record dataset with sub-second responsiveness while actively filtering and animating crime trends over time: the second of the project's main goals. It injects five data sources into the Kepler engine, including the raw crime points, the grocery store buffers, the enriched neighborhood polygons, and the final per-capita choropleth.
* Utilizes a `kepler_config.json` file to bypass manual UI setup. This ensures that the map—including custom color scales, 3D height settings, and the temporal animation slider—loads as intended every time the notebook is run.
* **GPU-Accelerated Rendering:** By handing the data to Kepler.gl, the project utilizes the user's graphics card to manage the heavy computation requirements of a large dataset. This allows for features that traditional GIS tools struggle with, such as:
    * Animating crime trends over the 2023–2026 timeline.
    * Using 3D extrusion and shaders to visualize crime metrics using vertical heights and color gradients.
* While displayed interactively within the Jupyter environment, the logic includes the ability to export a standalone, dependency-free `.html` file—ready for simple deployment to a static site hosting service.

**Live Interactive Kepler Maps:**
The code for generating Kepler maps within Jupyter has been disabled but the maps that can be viewed here:
* [Harm Per Capita Choropleth Map](https://harmpercapitachoropleth.netlify.app/)
* [Harm Density Time Series Animation](https://harmdensitytimeseries.netlify.app/)
* [All Generated Data](https://allcrimemapdata.netlify.app/)

**Conclusion:**
This project examined at the process of using a code-first geospatial approach (GeoPandas + Kepler.gl) to analyze and visualize large municipal datasets that would otherwise be too computationally expensive or credit-heavy for standard desktop GIS platforms. It provided a good introduction to the use of these tools and also the many challenges that arise when foregoing the use of more user-friendly GUI tools like ArcGIS.